In [ ]:
import numpy as np
import pandas as pd
import pydot

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, precision_score, balanced_accuracy_score, accuracy_score, recall_score, roc_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE, RFECV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from catboost import CatBoostClassifier

from collections import deque
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import trange


In [ ]:
import logging
import sys
from datetime import datetime

# Configure logging
log_filename = f"training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler(sys.stdout)  # optional: show logs live in notebook
    ]
)

logger = logging.getLogger(__name__)
logger.info("==== Test logging setup ====")


In [ ]:
# define constants according to your dataset

DATASET = "sample.csv"
COLUMN_NAME_LOS = "DurationHospStayDay"
COLUMN_NAME_ADMISSION_DATE = "DateAdmission"
COLUMN_NAME_TARGET = "Outcome"

In [ ]:
df = pd.read_csv('../data/' + DATASET, low_memory=False)
df = df.drop(columns=[COLUMN_NAME_LOS])
df.drop(COLUMN_NAME_ADMISSION_DATE, axis=1, inplace=True)

In [ ]:
# Example: binary outcome
X = df.drop(COLUMN_NAME_TARGET, axis=1)
y = df[COLUMN_NAME_TARGET]

In [ ]:
label_encoder = LabelEncoder()
label_mapping = {}
for column in X.columns:
  if X[column].dtype == 'object':
    X[column] = X[column].astype(str)
    X[column] = label_encoder.fit_transform(X[column])
    label_mapping[column] = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

logger.info(label_mapping.keys())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=123
)

In [ ]:
scoring = {
    "f1": make_scorer(f1_score, average="binary", pos_label="Dead"),
    "roc_auc": make_scorer(roc_auc_score, needs_proba=True),
    "precision": make_scorer(precision_score, pos_label="Dead"),
    "accuracy": make_scorer(accuracy_score),
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "recall": make_scorer(recall_score, pos_label="Dead")
}

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    res = {
        "f1" : f1_score(y_test, y_pred, average="binary", pos_label="Dead"),
        "roc_auc" : roc_auc_score(y_test, y_pred_proba),
        "precision" : precision_score(y_test, y_pred, pos_label="Dead"),
        "accuracy" : accuracy_score(y_test, y_pred),
        "balanced_accuracy" : balanced_accuracy_score(y_test, y_pred),
        "recall" : recall_score(y_test, y_pred, pos_label="Dead")
    }
    return res

## GBC

In [ ]:
gbc_pipeline = ImbPipeline(steps=[
    ('sampling', SMOTE(random_state=123)),
    ('model', GradientBoostingClassifier(
        learning_rate=0.01,
        max_depth=7, 
        max_features='sqrt', 
        min_samples_leaf=4, 
        min_samples_split=10, 
        n_estimators=100, 
        subsample=1.0
    ))])
selected_features = ['BW', 'GA', 'RespSpt_HFNCDone', 'RespSpt_HFOVDur', 'RespSpt_HFOVDone', 'Apgar1Min', 'MotherAge', 'RespSpt_NitrixOxideDone', 'RespSpt_NitrixOxideDur', 'PtRace_term', 'BirthAdmissionTemp', 'Surfactant_term', 'RespSpt_CPAPDone', 'Sex_term', 'SurfactantDuration', 'RespSpt_CPAPDur', 'Apgar5Min', 'BirthMultiplicity_term', 'RespSpt_ConvVentDur', 'InitResus_ETV_term']
gbc_pipeline.fit(X_train[selected_features], y_train)

In [ ]:
evaluate_model(gbc_pipeline, X_test[selected_features], y_test)

In [ ]:
print(gbc_pipeline.predict(X_test[selected_features].head(10)))
print(y_test.head(10).to_list())

## RF

In [ ]:
rf_pipeline = ImbPipeline(steps=[
    ('sampling', SMOTE(random_state=123)),
    ('model', RandomForestClassifier(
        criterion= 'log_loss', 
        max_depth= 20, 
        max_features= 'log2', 
        min_samples_leaf= 2, 
        min_samples_split= 2, 
        n_estimators= 100
    ))])
selected_features = ['BW', 'GA', 'MotherAge', 'RespSpt_HFNCDone', 'BirthAdmissionTemp', 'Apgar1Min', 'RespSpt_CPAPDur', 'RespSpt_HFOVDur', 'Apgar5Min', 'RespSpt_HFOVDone', 'RespSpt_CPAPDone', 'PtRace_term', 'SurfactantDuration', 'Parity', 'BirthFinalModeDeliver_term', 'Gravida', 'Sex_term', 'RespSpt_ConvVentDur', 'InitResus_ETV_term', 'BirthGrowthStatus_term', 'Surfactant_term', 'RespSpt_NitrixOxideDone', 'RespSpt_NitrixOxideDur', 'InitResus_CPAP_term', 'BirthMultiplicity_term', 'AtSteroid Dose_term', 'TPN', 'InitResus_O2_term', 'RespSpt_ConvVentDone', 'InitResus_BMV_term', 'AtSteroid_term', 'MaternalDM_term', 'MaternalHT_term', 'Intrapartum Antibiotic_term']
rf_pipeline.fit(X_train[selected_features], y_train)

In [ ]:
evaluate_model(rf_pipeline, X_test[selected_features], y_test)

## ETC

In [ ]:
etc_pipeline = ImbPipeline(steps=[
    ('sampling', SMOTE(random_state=123)),
    ('model', ExtraTreesClassifier(
        criterion='gini',
        max_depth=None,
        max_features='log2',
        min_samples_leaf=1,
        min_samples_split=2,
        n_estimators=100,
    ))])
selected_features = ['GA', 'RespSpt_HFNCDone', 'BW', 'Apgar1Min', 'RespSpt_HFOVDone', 'InitResus_CPAP_term', 'BirthGrowthStatus_term', 'RespSpt_ConvVentDur', 'RespSpt_CPAPDone', 'Surfactant_term', 'InitResus_ETV_term', 'RespSpt_CPAPDur', 'Sex_term', 'RespSpt_HFOVDur', 'MotherAge', 'Apgar5Min', 'RespSpt_ConvVentDone', 'PtRace_term', 'Parity', 'BirthAdmissionTemp', 'Gravida', 'SurfactantDuration', 'BirthFinalModeDeliver_term', 'AtSteroid_term', 'InitResus_BMV_term', 'AtSteroid Dose_term', 'TPN', 'MaternalDM_term', 'InitResus_O2_term', 'RespSpt_NitrixOxideDur', 'RespSpt_NitrixOxideDone', 'MaternalAnemia_term', 'BirthMultiplicity_term', 'Intrapartum Antibiotic_term', 'MaternalEclampsia_term', 'MaternalHT_term', 'RespSpt_term']
etc_pipeline.fit(X_train[selected_features], y_train)

In [ ]:
evaluate_model(etc_pipeline, X_test[selected_features], y_test)

## DTC

In [ ]:
dtc_pipeline = ImbPipeline(steps=[
    ('sampling', SMOTE(random_state=123)),
    ('model', DecisionTreeClassifier(
        criterion='entropy',
        max_depth=20,
        max_features='sqrt',
        min_samples_leaf=1,
        min_samples_split=5
    ))])
selected_features = ['BW', 'RespSpt_HFOVDone', 'RespSpt_HFNCDone', 'InitResus_ETV_term', 'RespSpt_CPAPDone', 'RespSpt_NitrixOxideDone', 'Parity']
dtc_pipeline.fit(X_train[selected_features], y_train)

In [ ]:
evaluate_model(dtc_pipeline, X_test[selected_features], y_test)

## CBC

In [ ]:
cbc_pipeline = ImbPipeline(steps=[
    ('sampling', SMOTE(random_state=123)),
    ('model', CatBoostClassifier(
        depth=4,
        iterations=500,
        l2_leaf_reg=3,
        learning_rate=0.1,
        verbose=0
    ))])
selected_features = ['RespSpt_HFOVDur', 'GA', 'Sex_term', 'RespSpt_HFNCDone', 'Apgar1Min', 'BW', 'RespSpt_NitrixOxideDone', 'MotherAge', 'PtRace_term', 'RespSpt_CPAPDur', 'RespSpt_NitrixOxideDur', 'BirthAdmissionTemp', 'SurfactantDuration', 'Apgar5Min', 'RespSpt_HFOVDone', 'Parity', 'BirthGrowthStatus_term', 'RespSpt_ConvVentDur', 'InitResus_ETV_term', 'Gravida', 'TPN', 'BirthFinalModeDeliver_term', 'BirthMultiplicity_term', 'AtSteroid Dose_term', 'RespSpt_CPAPDone', 'Surfactant_term', 'InitResus_O2_term', 'RespSpt_term', 'InitResus_CPAP_term', 'MaternalDM_term', 'InitResus_BMV_term', 'RespSpt_ConvVentDone', 'InitResus_CC_term', 'AtSteroid_term', 'MaternalAbPlacenta_term', 'MaternalEclampsia_term', 'MaternalCordProlapse_term']
cbc_pipeline.fit(X_train[selected_features], y_train)

In [ ]:
evaluate_model(cbc_pipeline, X_test[selected_features], y_test)

## Confidence Interval

In [ ]:
def bootstrap_metric(y_true, y_pred_or_prob, metric_fn, *,
                     n_bootstrap=1000, random_state=0, probs=False, stratified=True,
                     min_successful=1, max_attempts_factor=10):
    """
    Bootstrap CI for a metric.
    - metric_fn(y_true, y_pred_or_prob) should accept arrays.
    - probs=True: pass probabilities to metric_fn (eg roc_auc_score).
    - stratified=True: resample within each class to preserve class proportions (avoids single-class draws).
    Returns: mean, lower(2.5%), upper(97.5%), samples_array
    """
    rng = np.random.RandomState(random_state)
    y_true = np.asarray(y_true)
    n = len(y_true)
    if n == 0:
        raise ValueError("Empty y_true passed to bootstrap_metric")

    # encode labels to 0/1 for label-based metrics and roc_auc
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_bin = le.fit_transform(y_true)

    # prepare index sets per class for stratified sampling
    classes, class_idx = np.unique(y_bin, return_inverse=True)
    idx_by_class = {c: np.where(class_idx == c)[0] for c in classes}

    stats = []
    attempts = 0
    max_attempts = n_bootstrap * max_attempts_factor

    # convert input preds/probs once to numpy
    all_preds = np.asarray(y_pred_or_prob)

    while len(stats) < n_bootstrap and attempts < max_attempts:
        attempts += 1
        if stratified and len(classes) > 1:
            # sample within each class preserving original counts
            idx_list = []
            for c in classes:
                ids = idx_by_class[c]
                sampled = rng.choice(ids, size=len(ids), replace=True)
                idx_list.append(sampled)
            idx = np.concatenate(idx_list)
            rng.shuffle(idx)
        else:
            idx = rng.choice(n, n, replace=True)

        yt = y_bin[idx]
        yp = all_preds[idx]  # NOTE: use single indexing only

        # ensure both classes present when needed by metric (roc_curve/roc_auc)
        if probs:
            # roc_auc needs both classes and yp must be numeric probabilities
            if len(np.unique(yt)) < 2:
                continue
            try:
                yp_num = np.asarray(yp, dtype=float)
            except Exception:
                continue
            try:
                val = metric_fn(yt, yp_num)
            except Exception:
                continue
        else:
            # metric expects labels. If yp are probabilities, threshold at 0.5
            if np.issubdtype(np.asarray(yp).dtype, np.floating):
                yp_bin = (np.asarray(yp) >= 0.5).astype(int)
            else:
                # yp are label strings/ints — map to same encoding as y_true
                # use LabelEncoder fitted on y_true (le). If yp contains unknown labels, skip this resample.
                try:
                    yp_bin = le.transform(yp)
                except ValueError:
                    # unknown label present in predictions -> skip this bootstrap draw
                    continue

            # some metrics (like precision/recall) also require both classes present in yt or predictions;
            # we rely on metric_fn to raise if invalid, so catch exceptions below
            try:
                val = metric_fn(yt, yp_bin)
            except Exception:
                continue

        if np.isfinite(val):
            stats.append(val)

    if len(stats) < min_successful:
        raise RuntimeError(
            f"Insufficient successful bootstrap samples ({len(stats)}) after {attempts} attempts. "
            "Try fewer n_bootstrap, enable stratified sampling, or provide a larger/less imbalanced test set."
        )

    stats = np.array(stats)
    mean = float(stats.mean())
    lower = float(np.percentile(stats, 2.5))
    upper = float(np.percentile(stats, 97.5))
    return mean, lower, upper, stats

# wrapper metric functions that accept numeric 0/1 y
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             precision_score, balanced_accuracy_score, recall_score)

def auc_metric(y_true, y_prob): return roc_auc_score(y_true, y_prob)
def acc_metric(y_true, y_pred): return accuracy_score(y_true, y_pred)
def f1_metric(y_true, y_pred): return f1_score(y_true, y_pred)
def precision_metric(y_true, y_pred): return precision_score(y_true, y_pred)
def balacc_metric(y_true, y_pred): return balanced_accuracy_score(y_true, y_pred)
def recall_metric(y_true, y_pred): return recall_score(y_true, y_pred)

# helper to compute all requested metrics + CIs for a dataset
def compute_metrics_with_cis(model, X, y, *, n_bootstrap=1000, random_state=0, stratified=True):
    """
    Returns dict of metric -> (mean, lo, hi, samples).
    Metrics: F1, ROC AUC, Precision, Accuracy, Balanced Accuracy, Recall
    """
    # get probs and preds for full set
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = model.predict(X)

    metrics = {
        'roc_auc': (auc_metric, True),
        'f1': (f1_metric, False),
        'precision': (precision_metric, False),
        'accuracy': (acc_metric, False),
        'balanced_accuracy': (balacc_metric, False),
        'recall': (recall_metric, False),
    }

    results = {}
    for name, (fn, probs_flag) in metrics.items():
        arr = y_proba if probs_flag else y_pred
        mean, lo, hi, samples = bootstrap_metric(y, arr, fn,
                                                n_bootstrap=n_bootstrap,
                                                random_state=random_state,
                                                probs=probs_flag,
                                                stratified=stratified)
        results[name] = {'mean': mean, 'lo': lo, 'hi': hi, 'samples': samples}
    return results

# subgroup evaluation: returns DataFrame with CIs per subgroup for requested metrics
def evaluate_subgroups_with_cis(model, X_test, y_test, subgroup_col, *,
                                n_bootstrap=1000, min_subgroup_size=30, random_state=0, stratified=True):
    rows = []
    for grp_name, grp_df in X_test.groupby(subgroup_col):
        idx = grp_df.index
        if len(idx) < min_subgroup_size:
            continue
        X_sub = X_test.loc[idx]
        y_sub = y_test.loc[idx]
        try:
            res = compute_metrics_with_cis(model, X_sub, y_sub,
                                           n_bootstrap=n_bootstrap,
                                           random_state=random_state,
                                           stratified=stratified)
        except RuntimeError as e:
            # skip subgroups that cannot produce valid bootstrap samples
            rows.append({'subgroup': grp_name, 'n': len(idx), 'error': str(e)})
            continue

        row = {'subgroup': grp_name, 'n': len(idx)}
        for m, stats in res.items():
            row.update({
                f'{m}_mean': stats['mean'],
                f'{m}_lo': stats['lo'],
                f'{m}_hi': stats['hi']
            })
        rows.append(row)
    return pd.DataFrame(rows)

# convenience plotting for ROC with CI using stratified bootstrap by default
def roc_with_ci(y_true, y_proba, n_bootstrap=1000, random_state=0, stratified=True):
    """
    Plot mean ROC curve with 95% CI using stratified bootstrap to avoid empty-class resamples.
    """
    from sklearn.metrics import roc_curve
    rng = np.random.RandomState(random_state)
    base_fpr = np.linspace(0, 1, 101)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    # label-encode
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_bin = le.fit_transform(y_true)
    if len(np.unique(y_bin)) < 2:
        raise ValueError("y_true contains only one class; cannot compute ROC.")

    tprs = []
    attempts = 0
    max_attempts = n_bootstrap * 10

    classes, class_idx = np.unique(y_bin, return_inverse=True)
    idx_by_class = {c: np.where(class_idx == c)[0] for c in classes}

    while len(tprs) < n_bootstrap and attempts < max_attempts:
        attempts += 1
        if stratified:
            idx_list = []
            for c in classes:
                ids = idx_by_class[c]
                sampled = rng.choice(ids, size=len(ids), replace=True)
                idx_list.append(sampled)
            idx = np.concatenate(idx_list)
            rng.shuffle(idx)
        else:
            idx = rng.choice(len(y_bin), len(y_bin), replace=True)

        if len(np.unique(y_bin[idx])) < 2:
            continue
        try:
            fpr, tpr, _ = roc_curve(y_bin[idx], y_proba[idx])
            tprs.append(np.interp(base_fpr, fpr, tpr))
        except Exception:
            continue

    if len(tprs) == 0:
        raise RuntimeError("No valid bootstrap ROC samples; increase data or reduce n_bootstrap.")

    tprs = np.array(tprs)
    mean_tpr = tprs.mean(axis=0)
    lower_tpr = np.percentile(tprs, 2.5, axis=0)
    upper_tpr = np.percentile(tprs, 97.5, axis=0)

    plt.plot(base_fpr, mean_tpr, label=f"mean ROC (n={len(tprs)})")
    plt.fill_between(base_fpr, lower_tpr, upper_tpr, color="b", alpha=0.2, label="95% CI")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title("ROC with 95% CI")
    plt.legend()
    plt.show()

In [ ]:
y_proba = cbc_pipeline.predict_proba(X_test[selected_features])[:,1]
y_pred = cbc_pipeline.predict(X_test[selected_features])

mean_auc, low_auc, high_auc, auc_samples = bootstrap_metric(y_test, y_proba, auc_metric, n_bootstrap=1000, random_state=123, probs=True)
print(f"AUC: {mean_auc:.4f} (95% CI: {low_auc:.4f} - {high_auc:.4f})")

mean_f1, low_f1, high_f1, f1_samples = bootstrap_metric(y_test, y_pred, f1_metric, n_bootstrap=1000, random_state=123, probs=False)
print(f"F1 Score: {mean_f1:.4f} (95% CI: {low_f1:.4f} - {high_f1:.4f})")

mean_balacc, low_balacc, high_balacc, balacc_samples = bootstrap_metric(y_test, y_pred, balacc_metric, n_bootstrap=1000, random_state=123, probs=False)
print(f"Balanced Accuracy: {mean_balacc:.4f} (95% CI: {low_balacc:.4f} - {high_balacc:.4f})")

mean_precision, low_precision, high_precision, precision_samples = bootstrap_metric(y_test, y_pred, precision_metric, n_bootstrap=1000, random_state=123, probs=False)
print(f"Precision: {mean_precision:.4f} (95% CI: {low_precision:.4f} - {high_precision:.4f})")

mean_accuracy, low_accuracy, high_accuracy, accuracy_samples = bootstrap_metric(y_test, y_pred, acc_metric, n_bootstrap=1000, random_state=123, probs=False)
print(f"Accuracy: {mean_accuracy:.4f} (95% CI: {low_accuracy:.4f} - {high_accuracy:.4f})")

mean_recall, low_recall, high_recall, recall_samples = bootstrap_metric(y_test, y_pred, recall_metric, n_bootstrap=1000, random_state=123, probs=False)
print(f"Recall: {mean_recall:.4f} (95% CI: {low_recall:.4f} - {high_recall:.4f})")


In [ ]:
roc_with_ci(y_test, y_proba, n_bootstrap=1000)


In [ ]:
subgroup_table = evaluate_subgroups_with_cis(gbc_pipeline, X_test[selected_features], y_test, 'BW', n_bootstrap=1000)
display(subgroup_table)

## Clinician Demo

In [ ]:
import os
import numpy as np
from sklearn.tree import export_graphviz
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
import shutil

def export_gbc_trees(pipeline, feature_names, export_dir='gbc_tree_exports'):
    """
    Exports all individual trees from a GradientBoostingClassifier pipeline step
    to Graphviz .dot files in the specified directory.
    """
    if not os.path.exists(export_dir):
        os.makedirs(export_dir)
        print(f"Created directory: {os.path.abspath(export_dir)}")

    # Access the model step (User's pipeline uses 'model')
    try:
        model = pipeline.named_steps['model']
    except KeyError:
        print("Error: Could not find step named 'model' in pipeline.")
        return

    if not isinstance(model, GradientBoostingClassifier):
        print(f"Warning: The step 'model' is {type(model).__name__}, expected GradientBoostingClassifier.")
        return

    # exporters_ is an ndarray of shape (n_estimators, n_classes)
    # For binary classification, n_classes is 1.
    estimators = model.estimators_
    n_estimators = estimators.shape[0]
    n_classes = estimators.shape[1]

    print(f"Found {n_estimators} estimators (n_classes={n_classes}). Exporting...")

    generated_files = []

    for i in range(n_estimators):
        for j in range(n_classes):
            tree = estimators[i, j]
            
            # Construct filename
            filename = f"tree_{i:03d}.dot"
            filepath = os.path.join(export_dir, filename)
            
            # Export to .dot
            export_graphviz(
                tree,
                out_file=filepath,
                feature_names=feature_names,
                filled=True,
                rounded=True,
                special_characters=True,
                precision=4
            )
            generated_files.append(filepath)

    print(f"Successfully exported {len(generated_files)} .dot files.")
    return generated_files

In [ ]:
gbc_pipeline.named_steps['model'].estimators_

In [ ]:
gbc_selected_features = ['BW', 'GA', 'RespSpt_HFNCDone', 'RespSpt_HFOVDur', 'RespSpt_HFOVDone', 'Apgar1Min', 'MotherAge', 'RespSpt_NitrixOxideDone', 'RespSpt_NitrixOxideDur', 'PtRace_term', 'BirthAdmissionTemp', 'Surfactant_term', 'RespSpt_CPAPDone', 'Sex_term', 'SurfactantDuration', 'RespSpt_CPAPDur', 'Apgar5Min', 'BirthMultiplicity_term', 'RespSpt_ConvVentDur', 'InitResus_ETV_term']
export_gbc_trees(gbc_pipeline, gbc_selected_features, "tree_exports/gradient_boosting")

In [ ]:
import json
import numpy as np

def _to_native(val):
    return val.item() if isinstance(val, np.generic) else val

def collect_gbc_predictions(pipeline, X, y, feature_names):
    model = pipeline.named_steps['model']
    estimator_list = list(model.estimators_.ravel())
    learning_rate = model.learning_rate
    X_subset = X[feature_names].copy()
    y_subset = y.loc[X_subset.index]
    probabilities = pipeline.predict_proba(X_subset)[:, 1]

    payload = []
    for row_pos, (row_index, row) in enumerate(X_subset.iterrows()):
        row_array = row.values.reshape(1, -1)
        tree_entries = []
        for tree_idx, tree in enumerate(estimator_list):
            prediction = tree.predict(row_array)[0]
            contribution = learning_rate * prediction
            tree_entries.append({
                "index": int(tree_idx),
                "value": float(prediction),
                "logOdd": float(contribution),
                "dotPath": f"/dot/gradient_boosting/tree_{tree_idx:03d}.dot"
            })
        payload.append({
            "index": int(row_pos),
            "predictedProbability": float(probabilities[row_pos]),
            "predictedOutcome": _to_native(pipeline.predict(X_subset.iloc[[row_pos]])[0]),
            "actualOutcome": _to_native(y_subset.iloc[row_pos]),
            "inputFeatures": {col: _to_native(val) for col, val in row.items()},
            "trees": tree_entries
        })
    return payload

gbc_clinician_demo = collect_gbc_predictions(gbc_pipeline, X_test, y_test, gbc_selected_features)
print(json.dumps(gbc_clinician_demo, indent=2))

with open("GBC_Trees.json", 'w', encoding='utf-8') as json_file:
    json.dump(gbc_clinician_demo, json_file, indent=4)

In [ ]:
def export_tree_ensemble(pipeline, feature_names, export_dir):
    os.makedirs(export_dir, exist_ok=True)
    try:
        model = pipeline.named_steps['model']
    except KeyError:
        raise ValueError("Pipeline has no 'model' step.")
    estimators = getattr(model, "estimators_", None)
    if estimators is None:
        raise ValueError("Model has no estimators_.")
    trees = np.asarray(estimators).ravel()
    generated = []
    for idx, tree in enumerate(trees):
        filepath = os.path.join(export_dir, f"tree_{idx:03d}.dot")
        export_graphviz(
            tree,
            out_file=filepath,
            feature_names=feature_names,
            filled=True,
            rounded=True,
            special_characters=True,
            precision=4
        )
        generated.append(filepath)
    print(f"Exported {len(generated)} trees to {os.path.abspath(export_dir)}")
    return generated

rf_selected_features = ['BW', 'GA', 'MotherAge', 'RespSpt_HFNCDone', 'BirthAdmissionTemp',
                        'Apgar1Min', 'RespSpt_CPAPDur', 'RespSpt_HFOVDur', 'Apgar5Min',
                        'RespSpt_HFOVDone', 'RespSpt_CPAPDone', 'PtRace_term', 'SurfactantDuration',
                        'Parity', 'BirthFinalModeDeliver_term', 'Gravida', 'Sex_term',
                        'RespSpt_ConvVentDur', 'InitResus_ETV_term', 'BirthGrowthStatus_term',
                        'Surfactant_term', 'RespSpt_NitrixOxideDone', 'RespSpt_NitrixOxideDur',
                        'InitResus_CPAP_term', 'BirthMultiplicity_term', 'AtSteroid Dose_term',
                        'TPN', 'InitResus_O2_term', 'RespSpt_ConvVentDone', 'InitResus_BMV_term',
                        'AtSteroid_term', 'MaternalDM_term', 'MaternalHT_term',
                        'Intrapartum Antibiotic_term']

etc_selected_features = ['GA', 'RespSpt_HFNCDone', 'BW', 'Apgar1Min', 'RespSpt_HFOVDone',
                         'InitResus_CPAP_term', 'BirthGrowthStatus_term', 'RespSpt_ConvVentDur',
                         'RespSpt_CPAPDone', 'Surfactant_term', 'InitResus_ETV_term',
                         'RespSpt_CPAPDur', 'Sex_term', 'RespSpt_HFOVDur', 'MotherAge',
                         'Apgar5Min', 'RespSpt_ConvVentDone', 'PtRace_term', 'Parity',
                         'BirthAdmissionTemp', 'Gravida', 'SurfactantDuration',
                         'BirthFinalModeDeliver_term', 'AtSteroid_term', 'InitResus_BMV_term',
                         'AtSteroid Dose_term', 'TPN', 'MaternalDM_term', 'InitResus_O2_term',
                         'RespSpt_NitrixOxideDur', 'RespSpt_NitrixOxideDone', 'MaternalAnemia_term',
                         'BirthMultiplicity_term', 'Intrapartum Antibiotic_term',
                         'MaternalEclampsia_term', 'MaternalHT_term', 'RespSpt_term']

export_tree_ensemble(rf_pipeline, rf_selected_features, export_dir="tree_exports/random_forest")
export_tree_ensemble(etc_pipeline, etc_selected_features, export_dir="tree_exports/extra_trees")

In [ ]:
import json
import numpy as np

def collect_tree_ensemble_predictions(pipeline, X, y, feature_names, positive_label='Dead', folder_name=""):
    model = pipeline.named_steps['model']
    estimators = np.asarray(model.estimators_).ravel()
    classes = list(model.classes_)
    pos_idx = classes.index(positive_label) if positive_label in classes else (1 if len(classes) > 1 else 0)

    X_subset = X[feature_names].copy()
    y_subset = y.loc[X_subset.index]
    probabilities = pipeline.predict_proba(X_subset)[:, pos_idx]
    predicted_labels = pipeline.predict(X_subset)

    payload = []
    for row_pos, (_, row) in enumerate(X_subset.iterrows()):
        row_array = row.values.reshape(1, -1)
        tree_entries = []
        for tree_idx, tree in enumerate(estimators):
            prob = tree.predict_proba(row_array)[0, pos_idx]
            tree_entries.append({
                "index": int(tree_idx),
                "probability": float(prob),
                "dotPath": f"/dot/{folder_name}/tree_{tree_idx:03d}.dot"
            })
        payload.append({
            "index": int(row_pos),
            "predictedOutcome": _to_native(predicted_labels[row_pos]),
            "predictedProbability": float(probabilities[row_pos]),
            "actualOutcome": _to_native(y_subset.iloc[row_pos]),
            "inputFeatures": {col: _to_native(val) for col, val in row.items()},
            "trees": tree_entries
        })
    return payload

rf_clinician_demo = collect_tree_ensemble_predictions(
    rf_pipeline, X_test, y_test, rf_selected_features, positive_label='Dead', folder_name="random_forest"
)
etc_clinician_demo = collect_tree_ensemble_predictions(
    etc_pipeline, X_test, y_test, etc_selected_features, positive_label='Dead', folder_name="extra_trees"
)

print(json.dumps(rf_clinician_demo, indent=2))
print(json.dumps(etc_clinician_demo, indent=2))

with open("RF_Trees.json", "w", encoding="utf-8") as rf_file:
    json.dump(rf_clinician_demo, rf_file, indent=4)

with open("ETC_Trees.json", "w", encoding="utf-8") as etc_file:
    json.dump(etc_clinician_demo, etc_file, indent=4)